In [ ]:
import os
import json
import time
import pandas as pd
from mistralai import Mistral

# BILLING:
# https://docs.mistral.ai/getting-started/models/models_overview/


RESEARCHER = '.'
DATASET = 'correct-abstracts-500'
PROMPT = 'v1_balanced-prompt-without-examples'

MODEL_VERSION = 'mistral-large-latest'  # version 24.11 at the time of execution
# MODEL_VERSION = 'mistral-small-latest'

instruction_file = f'./{PROMPT}.txt'
dataset_file = f'./{DATASET}.xlsx'
output_file = f'./results/mistral_2025/{DATASET}__{PROMPT}__output.xlsx'

In [ ]:
api_key = os.environ["MISTRAL_API_KEY"]

client = Mistral(api_key=api_key)

df = pd.read_excel(dataset_file)

with open(instruction_file, 'r') as file:
    instructions = file.read()

# Print the instructions that will be fed to Mistral
print(instructions)

In [ ]:
def classify_paper(paper):
    
    chat_response = client.chat.complete(
        model=MODEL_VERSION,
        temperature= 0.0,
        response_format = {"type": "json_object"},
        messages = [
            {
                "role": "system",
                "content": instructions
            },
            {
                "role": "user",
                "content": paper
            }
        ]
    )
    
    # print(chat_response.choices[0].message.content)
    
    return chat_response.choices[0].message.content

In [ ]:
def update_output_file(output_file, new_row):

    # Define the columns for the DataFrame
    columns = ["ID", "Include/exclude", "Mistral verdict", "Mistral explanation", "Mistral confidence", "Title", "Authors", "Abstract"]
    
    # Check if the file exists
    if not os.path.exists(output_file):
        # Create a new DataFrame with the headers and save it to a new Excel file
        df = pd.DataFrame(columns=columns)

        # Append the new row to the DataFrame using concat
        df = pd.concat([df, new_row], ignore_index=True)
        
        df.to_excel(output_file, index=False)
        print(f"Created new file: {output_file} and added the first row.")
    else:
        # Load the existing file into a DataFrame
        df = pd.read_excel(output_file)

        # Append the new row to the DataFrame using concat
        df = pd.concat([df, new_row], ignore_index=True)
    
        # Save the updated DataFrame back to the Excel file
        df.to_excel(output_file, index=False)
        # print("New row added to the existing file.")

In [ ]:
true_positives = 0
true_negatives = 0
false_positives = 0
false_negatives = 0

# Record the start time
start_time = time.time()
total_rows_number = len(df)

# Iterate over the rows and print the content
for index, row in df.iterrows():

    r = row.to_dict()
    paper = f"Authors: {r['AUTHOR']}\n\nTitle: {r['TITLE']}\n\nAbstract: {r['ABSTRACT']}"
    # print(f"Paper ID {r['ID']}: {r['Title']}\n")
    print(f"Screening paper {index + 1}/{total_rows_number}")


    # Attempt to classify the paper and handle potential issues
    try:
        response_content = classify_paper(paper)
        # Attempt to parse the JSON response
        answer_dict = json.loads(response_content)
        
    except json.JSONDecodeError as e:
        # print(f"Failed to decode JSON response for paper ID {r['ID']}: {e}")
        # continue  # Skip this row and move to the next one if JSON parsing fails
        raise RuntimeError(f"Failed to decode JSON response for paper ID {r['ID']}: {e}")
    
    new_row = pd.DataFrame([{
        "ID": r['ID'],
        "Include/exclude": r['Include/exclude'],
        "Mistral verdict": answer_dict['verdict'],
        "Mistral explanation": answer_dict['explanation'],
        "Mistral confidence": answer_dict['confidence'],
        "Title": r['TITLE'],
        "Authors": r['AUTHOR'],
        "Abstract": r['ABSTRACT']
    }])

    update_output_file(output_file, new_row)

    if r['Include/exclude'] == answer_dict['verdict'] == "include":
        true_positives += 1
    elif r['Include/exclude'] == answer_dict['verdict'] == "exclude":
        true_negatives += 1
    elif r['Include/exclude'] == "include" and answer_dict['verdict'] == "exclude":
        false_negatives += 1
    elif r['Include/exclude'] == "exclude" and answer_dict['verdict'] == "include":
        false_positives += 1
        

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"\n\nThis paper screening took {elapsed_time:.1f} seconds")
print(f"The screening included {true_positives + true_negatives + false_positives + false_negatives} papers, out of which {true_positives + true_negatives} were correctly classified")
print(f"True positives: {true_positives}")
print(f"True negatives: {true_negatives}")
print(f"False positives: {false_positives}")
print(f"False negatives: {false_negatives}")
